In [0]:
# --------------------------------------------------------------------------------------------------
# SCRIPT: 1.1_atlas_frentes_consolidado
# LOCAL: 3_gold/src/1_atlas_frentes/
# OBJETIVO: Gerar tabelas de Frentes e seus membros usando como fonte tabelas dos scripts 
#           /2_silver/src/1_atlas_frentes/2_membros_frentes_transform e 1.1_frentes_transform
# ENTREGA: Tabela gold de frentes com membros, partido, UF e legislatura por frente
# --------------------------------------------------------------------------------------------------

from pyspark.sql.functions import col, count

# 1. Carregar as tabelas Silver
df_frentes = spark.table("silver_frentes")
df_membros = spark.table("silver_membros_frentes")

# 2. Criar o Atlas Consolidado fazendo o join entre a definição da frente e seus integrantes
df_atlas = df_membros.join(df_frentes, df_membros.id_frente == df_frentes.id, "inner") \
    .select(
        col("id_frente"),
        col("titulo").alias("nome_frente"),
        col("id_deputado"),
        col("nome_deputado"),
        col("partido"), 
        col("uf"),
        col("idLegislatura").alias("id_legislatura")
    ) \
    .filter(col("partido").isNotNull())

# 3. Salvar na Gold
df_atlas.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_atlas_frentes_parlamentares")

# 4. Visualizar resultado
display(df_atlas)

In [0]:
df_frente = spark.table("gold_atlas_frentes_parlamentares")
# group by
df_frente.groupBy("nome_frente").count().show()
 
df_ranking = df_frente \
    .groupBy("nome_frente").count() 

 
df_ranking.orderBy("count").show(10)
 
df_ranking.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_atlas_frentes_parlamentares_ranking") 


 

In [0]:
spark.sql("Drop table if exists gold_atlas_frentes_parlamentares_ranking")

In [0]:
display(spark.table("gold_atlas_frentes_parlamentares")
        .filter("nome_frente LIKE '%Agro%' OR nome_frente LIKE '%Ambiente%'")
        .select("nome_frente").distinct())